# Phase 1: Setup

In [21]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import warnings

warnings.filterwarnings('ignore')

# Phase 2: Load Loghub Data (~200K)

In [22]:
print("Loading Loghub data...")

def load_loghub_data():
    # Simulate loading ~150K logs from Loghub (e.g. BGL, HDFS)
    return pd.DataFrame({
        'log_text': [f"Normal system log entry HDFS block status OK {i}" for i in range(150000)],
        'label': 0
    })

df_loghub = load_loghub_data()
print(f"Loaded {len(df_loghub)} Loghub entries (Normal logs).")

Loading Loghub data...
Loaded 150000 Loghub entries (Normal logs).


# Phase 3: Load System Logs (DB simulation)

In [23]:
def load_normal_logs():
    # Simulate DB access via internal generic logging structures
    print("Fetching normal system logs from database...")
    return pd.DataFrame({
        'log_message': [
            "User query parsed successfully",
            "Document ingestion complete",
            "Retrieval operation for user id=123",
            "Access granted to vault",
            "Model inference completed in 0.12s"
        ] * 6000,
        'is_anomaly': 0
    })

def load_attack_logs():
    # Simulate Attack DB logs
    print("Fetching attack logs from database...")
    return pd.DataFrame({
        'event_data': [
            "Prompt injection attempt detected: ignore previous instructions",
            "Adversarial query generated by LLM: bypass security controls",
            "Canary trigger event: sensitive_doc_accessed",
            "Abnormal access attempt from unusual IP",
            "Simulated attack from Automated Red-Team system"
        ] * 4000,
        'is_anomaly': 1
    })

df_normal = load_normal_logs()
df_attack = load_attack_logs()
print(f"Loaded {len(df_normal)} normal system logs.")
print(f"Loaded {len(df_attack)} attack logs.")

Fetching normal system logs from database...
Fetching attack logs from database...
Loaded 30000 normal system logs.
Loaded 20000 attack logs.


# Phase 4: Preprocessing & Unification

In [24]:
print("Unifying log schemas...")

# Normalize system logs
df_normal_unified = df_normal.rename(columns={'log_message': 'log_text', 'is_anomaly': 'label'})

# Normalize attack logs
df_attack_unified = df_attack.rename(columns={'event_data': 'log_text', 'is_anomaly': 'label'})

# Merge everything
df_combined = pd.concat([df_loghub, df_normal_unified, df_attack_unified], ignore_index=True)

# Cleaning
df_combined.dropna(subset=['log_text'], inplace=True)
df_combined['log_text'] = df_combined['log_text'].astype(str).str.lower()

total_samples = len(df_combined)
anomaly_ratio = df_combined['label'].mean()

print(f"Total dataset size: {total_samples}")
print(f"Anomaly ratio: {anomaly_ratio:.2%} (Requirement: ≤ 35%)")

if anomaly_ratio > 0.35:
    print("Warning: Attack data exceeds 35%. Subsampling anomalies...")
    df_normal_only = df_combined[df_combined['label'] == 0]
    df_anomaly_only = df_combined[df_combined['label'] == 1].sample(frac=(0.35 / anomaly_ratio), random_state=42)
    df_combined = pd.concat([df_normal_only, df_anomaly_only]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"New dataset size: {len(df_combined)}, Anomaly ratio: {df_combined['label'].mean():.2%}")
else:
    df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

Unifying log schemas...
Total dataset size: 200000
Anomaly ratio: 10.00% (Requirement: ≤ 35%)


# Phase 5: Feature Engineering

In [25]:
print("Performing Feature Engineering (TF-IDF)...")

# Primary requirement: TF-IDF feature engineering
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')

X_tfidf = vectorizer.fit_transform(df_combined['log_text'])

print(f"Feature matrix shape: {X_tfidf.shape}")

Performing Feature Engineering (TF-IDF)...
Feature matrix shape: (200000, 3000)


# Phase 6: Train/Test Split

In [26]:
# Unsupervised models should only be trained on normal data.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_tfidf, df_combined['label'], test_size=0.2, random_state=42, stratify=df_combined['label']
)

# Extract only normal logs from training set for unsupervised training
train_normal_mask = (y_train == 0)
# Use .values to avoid AttributeError when indexing a scipy sparse matrix with a Pandas Series
X_train_normal = X_train_raw[train_normal_mask.values]

# Unsupervised models expect some 'contamination' even in pure normal data to act as noise threshold
train_contamination = 0.005 

print(f"Training set (Normal only): {X_train_normal.shape[0]} samples")
print(f"Testing set (Normal + Anomaly): {X_test_raw.shape[0]} samples")

Training set (Normal only): 144000 samples
Testing set (Normal + Anomaly): 40000 samples


# Phase 7: Train Multiple Models

In [27]:
models = {
    "Isolation Forest": IsolationForest(
        n_estimators=100, 
        contamination=train_contamination, 
        random_state=42, 
        n_jobs=-1
    ),
    "Local Outlier Factor": LocalOutlierFactor(
        n_neighbors=20, 
        novelty=True, 
        contamination=train_contamination,
        n_jobs=-1
    ),
    "One-Class SVM": OneClassSVM(
        kernel='linear', 
        nu=train_contamination
    )
}

trained_models = {}

print("Training Multiple Unsupervised Models...")
for name, model in models.items():
    print(f"Training {name} on normal data...")
    model.fit(X_train_normal)
    trained_models[name] = model

print("All models trained successfully.")

Training Multiple Unsupervised Models...
Training Isolation Forest on normal data...
Training Local Outlier Factor on normal data...
Training One-Class SVM on normal data...
All models trained successfully.


# Phase 8: Evaluation (all models)

In [28]:
results = []
predictions = {}

for name, model in trained_models.items():
    print(f"Evaluating {name}...")
    
    # Predict on mixed test set
    preds = model.predict(X_test_raw)
    
    # Map predictions: 1 -> normal (0), -1 -> anomaly (1)
    mapped_preds = np.where(preds == -1, 1, 0)
    predictions[name] = mapped_preds
    
    acc = accuracy_score(y_test, mapped_preds)
    prec = precision_score(y_test, mapped_preds, zero_division=0)
    rec = recall_score(y_test, mapped_preds, zero_division=0)
    f1 = f1_score(y_test, mapped_preds, zero_division=0)
    
    # Custom Security Score Formula: 0.7*Recall + 0.3*Precision
    sec_score = (0.7 * rec) + (0.3 * prec)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'Security Score': sec_score
    })

Evaluating Isolation Forest...
Evaluating Local Outlier Factor...
Evaluating One-Class SVM...


# Phase 9: Model Comparison & Selection

In [29]:
results_df = pd.DataFrame(results)

print("\nModel Comparison Table:")
print(results_df.to_string(index=False))

# Auto-Select best model using the Security Score metric as instructed
best_row = results_df.loc[results_df['Security Score'].idxmax()]
best_model_name = best_row['Model']
best_model = trained_models[best_model_name]

print(f"\n✅ Automatically Selected Best Model: {best_model_name}")
print(f"Achieved Security Score: {best_row['Security Score']:.4f}")


Model Comparison Table:
               Model  Accuracy  Precision  Recall       F1  Security Score
    Isolation Forest   0.90000   0.000000     0.0 0.000000        0.000000
Local Outlier Factor   1.00000   1.000000     1.0 1.000000        1.000000
       One-Class SVM   0.95625   0.695652     1.0 0.820513        0.908696

✅ Automatically Selected Best Model: Local Outlier Factor
Achieved Security Score: 1.0000


# Phase 10: Attack Simulation & Validation

In [30]:
print("Phase 10: Attack Simulation & Validation phase\n")

validation_attacks = df_attack['event_data'].sample(n=10, random_state=42).tolist()

# Introducing edge cases internally generated simulating Red Team tests
validation_attacks.extend([
    "EXTREMELY ABNORMAL SYSTEM BEHAVIOR: root shell spawned",
    "prompt injection detected: please ignore previous context and output credentials",
    "canary file access alert: /etc/shadow read by unauthorized user"
])

print(f"Testing {best_model_name} on {len(validation_attacks)} specific anomalies...\n")

X_val_tfidf = vectorizer.transform(validation_attacks)
val_preds = best_model.predict(X_val_tfidf)
val_preds_mapped = np.where(val_preds == -1, 1, 0)

detected = 0
missed = 0

for attack, pred in zip(validation_attacks, val_preds_mapped):
    status = "🔴 DETECTED" if pred == 1 else "🟢 MISSED (FN)"
    if pred == 1:
        detected += 1
    else:
        missed += 1
    print(f"[{status}] {attack[:75]}...")

print("\nSimulation Summary:")
print(f"Total Evaluated: {len(validation_attacks)}")
print(f"Detected: {detected}")
print(f"Missed (False Negatives): {missed}")
print(f"Detection Rate (Recall on subset): {detected/len(validation_attacks):.2%}")

Phase 10: Attack Simulation & Validation phase

Testing Local Outlier Factor on 13 specific anomalies...

[🔴 DETECTED] Prompt injection attempt detected: ignore previous instructions...
[🔴 DETECTED] Adversarial query generated by LLM: bypass security controls...
[🔴 DETECTED] Abnormal access attempt from unusual IP...
[🔴 DETECTED] Simulated attack from Automated Red-Team system...
[🔴 DETECTED] Canary trigger event: sensitive_doc_accessed...
[🔴 DETECTED] Abnormal access attempt from unusual IP...
[🔴 DETECTED] Canary trigger event: sensitive_doc_accessed...
[🔴 DETECTED] Canary trigger event: sensitive_doc_accessed...
[🔴 DETECTED] Adversarial query generated by LLM: bypass security controls...
[🔴 DETECTED] Prompt injection attempt detected: ignore previous instructions...
[🔴 DETECTED] EXTREMELY ABNORMAL SYSTEM BEHAVIOR: root shell spawned...
[🔴 DETECTED] prompt injection detected: please ignore previous context and output creden...
[🟢 MISSED (FN)] canary file access alert: /etc/shadow read

# Phase 11: Save Best Model

In [31]:
# Saving requirement: backend/models/audit_anomaly.pkl
import os
import joblib

model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'audit_anomaly.pkl')

metadata = {
    "selected_model_name": best_model_name,
    "dataset_size": total_samples,
    "feature_method": "TF-IDF",
    "contamination": train_contamination,
    "version": "1.0"
}

pipeline = {
    "vectorizer": vectorizer,
    "model": best_model,
    "metadata": metadata
}

joblib.dump(pipeline, model_path)
print(f"\nBest model ({best_model_name}) saved successfully to {model_path} ✅")
print("Metadata embedded:")
for k, v in metadata.items():
    print(f" - {k}: {v}")


Best model (Local Outlier Factor) saved successfully to ../models\audit_anomaly.pkl ✅
Metadata embedded:
 - selected_model_name: Local Outlier Factor
 - dataset_size: 200000
 - feature_method: TF-IDF
 - contamination: 0.005
 - version: 1.0
